# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E001-pipe-check-gold58** — three per-plane fluid-sensitive
models (frozen backbone + linear head) on the 58 gold-labeled studies (issue #6).
Requires the `WANDB_API_KEY` Kaggle secret.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
COMMIT = "4ef6afc"  # main @ PR #8 squash: per-plane training + ensemble inference
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_gold import train_gold

In [ ]:
# Gold-58 prototype trains straight off the mounted competition data; the private
# mined-labels dataset joins here later (issue #2).
from pathlib import Path

SLUG = "rsna-knee-abnormality-detection"
# Kaggle mounts competitions under /kaggle/input/competitions/<slug> (newer layout)
# or /kaggle/input/<slug> (older docs/examples); accept either.
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(project="rsna-knee", config={"commit": COMMIT})
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E001: one specialist per fluid plane (strict typing — no fallback series; studies
# lacking a plane are skipped by that model). Non-fluid models are a contingency,
# not a plan — see docs/modeling understanding/optimization-levers.md.
SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
INPUT_SIZE = 224

# pipe_check_gold58: trains on the gold studies, must never be evaluated against them.
def checkpoint_path(series_type: SeriesType) -> Path:
    return Path(f"/kaggle/working/pipe_check_gold58_{series_type.value}.pt")

In [ ]:
# Competition DICOMs are pre-mounted read-only; ~58 series per model, minutes of GPU each.
results = []
for series_type in SERIES_TYPES:
    result = train_gold(COMP_ROOT, checkpoint_path(series_type), series_type=series_type, input_size=INPUT_SIZE)
    results.append(result)
    print(f"{series_type.value}: trained on {result.n_studies}, skipped {len(result.skipped)}")
    # In-sample only (trains on all gold rows): proves the features carry signal, nothing more.
    print(result.in_sample_auc)

In [ ]:
# Local eval (DECISIONS.md #4): pooled out-of-fold stratified CV of the full ensemble
# on the gold studies — each study predicted by heads that never saw it, merged by the
# production combiner. This is the Val AUC recorded in experiments.md; the shipped
# checkpoints above still train on all gold rows. Features extract once (frozen
# backbone), so 25 head fits add only seconds. Needs COMMIT >= the cv_gold merge.
from knee.cv_gold import collect_gold_features, cross_validate_gold

bank = collect_gold_features(COMP_ROOT, series_types=SERIES_TYPES, input_size=INPUT_SIZE)
cv = cross_validate_gold(bank)
print("plane coverage:", {t.value: n for t, n in cv.plane_coverage.items()})
print(f"macro OOF AUC {cv.macro_auc:.3f} over {cv.n_repeats} repeats: "
      + ", ".join(f"{m:.3f}" for m in cv.macro_auc_per_repeat))
print({label: round(auc, 3) for label, auc in cv.per_label_auc.items()})

In [ ]:
# Checkpoints are already in /kaggle/working, which persists as notebook output;
# publish all three as the knee-weights dataset so the inference notebook can attach them.
import math

if run is not None:
    for result in results:
        wandb.log(
            {
                f"in_sample_auc/{result.series_type.value}/{label}": auc
                for label, auc in result.in_sample_auc.items()
                if not math.isnan(auc)
            }
        )
    wandb.log(
        {
            "cv/macro_auc": cv.macro_auc,
            **{f"cv/auc/{label}": auc for label, auc in cv.per_label_auc.items() if not math.isnan(auc)},
        }
    )
    run.finish()